<a href="https://colab.research.google.com/github/nikitask14/adult-income-classification/blob/main/adult_income_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


*  Importing Pandas, numpy library to structure the raw data into readable tables (DataFrames) and handle numerical operations.
*  Importing the skicit library - sklearn and importing the dataset loader - fetch_openml




In [263]:
from sklearn.datasets import fetch_openml
import pandas as pd
import numpy as np

### Dataset loading note

The original plan was to load the Adult dataset using `fetch_openml`, but OpenML repeatedly returned a 504 Gateway Timeout.

To avoid delaying the project, I loaded the same Adult/Census Income dataset from the UCI Machine Learning Repository using `ucimlrepo`.



In [264]:
!pip install ucimlrepo


In [265]:
from ucimlrepo import fetch_ucirepo

adult = fetch_ucirepo(id=20)

X = adult.data.features
y = adult.data.targets

After loading:
- `X` is a Pandas DataFrame with shape `(48842, 14)`
- `y` should be a 1D Pandas Series with shape `(48842,) but the UCI loader initially returned the target as a one-column DataFrame of shape `(48842, 1), so the target converted to a Series before continuing.

In [266]:
print(type(X))
print(X.shape)


<class 'pandas.core.frame.DataFrame'>
(48842, 14)


In [267]:
print(type(y))
print(y.shape)


<class 'pandas.core.frame.DataFrame'>
(48842, 1)


In [268]:
y.columns

Index(['income'], dtype='object')

In [269]:
y = y["income"]



In [270]:
type(y)
y.shape

(48842,)

Using info() method to get a comprehensive summary about the dataset.


1.  It tells us about the number of rows and columns.
2.  How many missing values are present
3.  What the numerical and categorical features are.
4.  The Non-Null tells us the number of actual values present for each of the feature columns.
5.  Columns with 48,842 non-null (like age or education) are completely full.



In [271]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       47879 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education-num   48842 non-null  int64 
 5   marital-status  48842 non-null  object
 6   occupation      47876 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital-gain    48842 non-null  int64 
 11  capital-loss    48842 non-null  int64 
 12  hours-per-week  48842 non-null  int64 
 13  native-country  48568 non-null  object
dtypes: int64(6), object(8)
memory usage: 5.2+ MB


Now, checking the number of null values in for each of the feature columns and sorting them by features with highest missing values to lowest.

In [272]:
X.isnull().sum().sort_values(ascending = False)

,0
occupation,966
workclass,963
native-country,274
education,0
fnlwgt,0
age,0
marital-status,0
education-num,0
race,0
relationship,0


occupation has 2809 missing values, workclass	has 2799 missing values, native-country	has 857 missing values.





In [273]:
X.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba


 DATA AUDIT  
*   Check for class imbalance
*   fnlwgt(doesn't seem to carry a lot of meaning), marital status not of much consequnce if earning or not, Race is controversial,
*   Check for data leakage - found none upfront.
*  Education and education-num seem similar - maybe keep education-num and drop out education to save dimensionality.








In [274]:
X["native-country"].unique()

array(['United-States', 'Cuba', 'Jamaica', 'India', '?', 'Mexico',
       'South', 'Puerto-Rico', 'Honduras', 'England', 'Canada', 'Germany',
       'Iran', 'Philippines', 'Italy', 'Poland', 'Columbia', 'Cambodia',
       'Thailand', 'Ecuador', 'Laos', 'Taiwan', 'Haiti', 'Portugal',
       'Dominican-Republic', 'El-Salvador', 'France', 'Guatemala',
       'China', 'Japan', 'Yugoslavia', 'Peru',
       'Outlying-US(Guam-USVI-etc)', 'Scotland', 'Trinadad&Tobago',
       'Greece', 'Nicaragua', 'Vietnam', 'Hong', 'Ireland', 'Hungary',
       'Holand-Netherlands', nan], dtype=object)

In [275]:
X["occupation"].unique()

array(['Adm-clerical', 'Exec-managerial', 'Handlers-cleaners',
       'Prof-specialty', 'Other-service', 'Sales', 'Craft-repair',
       'Transport-moving', 'Farming-fishing', 'Machine-op-inspct',
       'Tech-support', '?', 'Protective-serv', 'Armed-Forces',
       'Priv-house-serv', nan], dtype=object)

The UCI loader preserved some values differently from OpenML. Missing categorical values appeared party as "?", and party as NaN. I corrected this so that all missing values show as NaN.

In [276]:
X = X.replace("?",np.nan)
X["occupation"].unique()

array(['Adm-clerical', 'Exec-managerial', 'Handlers-cleaners',
       'Prof-specialty', 'Other-service', 'Sales', 'Craft-repair',
       'Transport-moving', 'Farming-fishing', 'Machine-op-inspct',
       'Tech-support', nan, 'Protective-serv', 'Armed-Forces',
       'Priv-house-serv'], dtype=object)

In [277]:
y.value_counts()

,count
income,
<=50K,24720
<=50K.,12435
>50K,7841
>50K.,3846


The target appeared as four strings variants because some labels had trailing periods, and to correct this, we need to strip the trailing periods from the category names so the duplicate rows merge together.

Day 3 - Baseline Prediction: Since the class labels were in the form of <=50K, >50K, so we do the following:

0 = <=50K
1 = >50K

This allows are prediction class(0 or 1) and actual class labels to be of the same type.

In [278]:
y = y.str.rstrip('.')
y = y.map({"<=50K": 0, ">50K": 1})
y.value_counts()

,count
income,
0,37155
1,11687


Moderate class imbalance

---


Class 0 = 76%
Class 1 = 24%

Because the classes are imbalanced, all train/validation/test splits are stratified to preserve approximately the same class proportions.

In [279]:
retained_columns = ["occupation", "workclass", "native-country", "age",	"marital-status", "education-num", "race",
                    "relationship", "sex", "capital-gain", "capital-loss", "hours-per-week"]

In [280]:
type(retained_columns)

list

In [281]:
X_selected = X[retained_columns]
X_selected.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   occupation      46033 non-null  object
 1   workclass       46043 non-null  object
 2   native-country  47985 non-null  object
 3   age             48842 non-null  int64 
 4   marital-status  48842 non-null  object
 5   education-num   48842 non-null  int64 
 6   race            48842 non-null  object
 7   relationship    48842 non-null  object
 8   sex             48842 non-null  object
 9   capital-gain    48842 non-null  int64 
 10  capital-loss    48842 non-null  int64 
 11  hours-per-week  48842 non-null  int64 
dtypes: int64(5), object(7)
memory usage: 4.5+ MB


We now have 48842 entries.

> 12 feature columns, positive class is income >$50. High class imbalance, no leakage, raw/high cardinality features removed. Redundant features removed.



In [282]:
from sklearn.model_selection import train_test_split

Next, we are creating a training test split and then diving the training split further to create a validation set to decide on the how well the model performs on unseen data before finally checking on how the model performs on test(unseen) data.

It also ensures there is no data leakage.

In [283]:
X_train_val, X_test_final, y_train_val, y_test_final = train_test_split(
    X_selected,
    y,
    test_size = 0.20,
    random_state = 42,
    stratify = y
)

In [284]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size = 0.25,
    random_state=42,
    stratify = y_train_val

)

Class 0 = 76%
Class 1 = 24%

Because the classes are imbalanced, all train/validation/test splits are stratified to preserve approximately the same class proportions.

In [285]:
print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)
print(X_test_final.shape)
print(y_test_final.shape)
print(type(y_val))

(29304, 12)
(29304,)
(9769, 12)
(9769,)
(9769, 12)
(9769,)
<class 'pandas.core.series.Series'>


Now, that we have split the data, we will now identify the numerical and categorical columns before preprocessing as they require different preprocessing techniques.

In [286]:
numerical_columns = X_selected.select_dtypes(include = "number").columns
numerical_columns
# len(numerical_columns)

Index(['age', 'education-num', 'capital-gain', 'capital-loss',
       'hours-per-week'],
      dtype='object')

In [287]:
categorical_columns = X_selected.select_dtypes(include = "object").columns
categorical_columns
# len(categorical_columns)


Index(['occupation', 'workclass', 'native-country', 'marital-status', 'race',
       'relationship', 'sex'],
      dtype='object')

In [288]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (StandardScaler, OneHotEncoder)
from sklearn.compose import ColumnTransformer


In [289]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy = "median")),
    ("scale", StandardScaler())
])

In [290]:
cat_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy = "most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [291]:
preprocessor = ColumnTransformer(
    [
        ("numerical",num_pipeline, numerical_columns),
        ("cat", cat_pipeline, categorical_columns)
    ]
)

We now create a baseline to check if our model would world better than if we did nothing. We will create a baseline prediction and perform evaluation using Accuracy, Precision, Recall and F1 score. We will also compute the confusion matrix.

We have created a new array of same length as y_val and have set the values to zero as the majority labels are Class 0 labels.

The baseline predicts majority class determined from y_train, not from validation or test targets.

In [292]:
baseline_predictions = np.full(len(y_val),0)
baseline_predictions
np.unique(baseline_predictions)
# len(baseline_prediction)

array([0])

In [293]:
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, accuracy_score, f1_score)

In [294]:
baseline_cm = confusion_matrix(y_val, baseline_predictions)
baseline_cm

array([[7432,    0],
       [2337,    0]])

In [295]:
baseline_accuracy = accuracy_score(y_val, baseline_predictions)
baseline_accuracy

0.7607738765482649

In [296]:
baseline_precision = precision_score(y_val, baseline_predictions)
baseline_precision

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


0.0

In [297]:
baseline_recall = recall_score(y_val, baseline_predictions)
baseline_recall

0.0

In [298]:
baseline_f1_score = f1_score(y_val, baseline_predictions)
baseline_f1_score

0.0

**Baseline Results**: The majority-class baseline achieves about 76% accuracy by predicting <=50K for every observation, but its precision, recall and F1-Score for the positive class are 0 because it never predicts >50K.

In [299]:
from sklearn.linear_model import LogisticRegression

In [300]:
model_pipeline = Pipeline([
    ("preprocessor", preprocessor ),
    ("regression", LogisticRegression(max_iter = 1000))
])
model_pipeline




Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  Index(['age', 'education-num', 'capital-gain', 'capital-loss',
       'hours-per-week'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['occupation', 'workclass', 'native-country', 'marital-status', 'race',
       'relationship', 'sex'],
      dtype='object'))])),
                ('regression', LogisticRegression(max_iter=1000))])

In [301]:
model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  Index(['age', 'education-num', 'capital-gain', 'capital-loss',
       'hours-per-week'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['occupation', 'workclass', 'native-country', 'marital-status', 'race',
       'relationship', 'sex'],
      dtype='object'))])),
                ('regression', LogisticRegression(max_iter=1000))])

In [319]:
val_prob = model_pipeline.predict_proba(X_val)

print(val_prob)
print()
print(val_prob.shape)

<class 'numpy.ndarray'>
[[0.96628015 0.03371985]
 [0.39587686 0.60412314]
 [0.30732091 0.69267909]
 ...
 [0.99747771 0.00252229]
 [0.89316475 0.10683525]
 [0.86515634 0.13484366]]

(9769, 2)


(9796, 2)

because there are:

9796 validation adults; 2 probability columns, one for each class.

The columns mean:

column 0 → P(class 0) = P(<=50K)

column 1 → P(class 1) = P(>50K)



In [303]:
model_pipeline.classes_

array([0, 1])

In [310]:
val_pred = model_pipeline.predict(X_val)
print(val_pred)
print()
print(val_pred.shape)

[0 1 1 ... 0 0 0]

(9769,)


In [312]:
val_cm = confusion_matrix(y_val, val_pred)
val_cm

array([[6880,  552],
       [ 895, 1442]])

In [313]:
val_accuracy = accuracy_score(y_val, val_pred)
val_accuracy

0.8518783908281298

In [314]:
val_precision = precision_score(y_val, val_pred)
val_precision

0.7231695085255767

In [315]:
val_recall = recall_score(y_val, val_pred)
val_recall

0.6170303808301241

In [317]:
val_f1_score = f1_score(y_val, val_pred)
val_f1_score

0.6658970214731009

**Supported Conclusion:**


Logistic Regression is more useful than predicting the majority class for all earners.

It improves the overall accuracy from 76% to 85.1%.

When the model predicts that the earner earns >$50K, it is right 72.31% of the times.

Of all the earner earning more than $50K, it is right for 66.58% of the times.

The model is still not perfect: It still misses about 38% of the actual >$50K earners which corresponds to 895 False Negatives.

It also misclassifies 552 earners who earn 50K or  or less than 50K as earning more than $50K, which are False Positives.

00 - TN - 6880
01 - FP - 552
10 - FN - 895
11 - TP - 1442



Now we conduct failure analysis on the Validation Set.  

We start by getting class 1 probabilities as this the class we want to predict i.e., earners with income >$50K.




In [325]:
class1_probabilities = val_prob[:,1]

Now, predict_prob gives a 2D numpy array with the probbaility of class 0 and class 1 prediction. Since we are interested in Class 1, that is the class that we want to predict, we do a [;, 1] which gives us all rows, 2nd column.

In [326]:
class1_probabilities.shape

(9769,)

We now create a copy of X_val and add three additional columns for feature analysis, which are - actual labels, predicted labels and class1_probabilities(certainity of prediction). This will later also allow to freeze the threshold.

In [327]:
failure_analysis_table = X_val.copy()
failure_analysis_table["actual_label"] = y_val
failure_analysis_table["predicted_label"] = val_pred
failure_analysis_table["class 1 Probabilities"] = class1_probabilities

In [328]:
failure_analysis_table

,occupation,workclass,native-country,age,marital-status,education-num,race,relationship,sex,capital-gain,capital-loss,hours-per-week,actual_label,predicted_label,class 1 Probabilities
27033,Sales,Private,China,36,Never-married,13,Asian-Pac-Islander,Not-in-family,Female,0,0,50,0,0,0.033720
36320,Prof-specialty,Private,United-States,36,Married-civ-spouse,13,White,Husband,Male,0,0,40,1,1,0.604123
34540,Prof-specialty,Local-gov,United-States,51,Married-civ-spouse,14,White,Husband,Male,0,0,35,0,1,0.692679
48723,Sales,Private,United-States,48,Married-civ-spouse,13,White,Husband,Male,0,0,45,1,1,0.643265
18991,Protective-serv,Private,United-States,24,Divorced,9,White,Own-child,Female,0,1762,40,0,0,0.021968
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46693,Sales,Private,United-States,30,Never-married,12,Black,Unmarried,Female,0,0,40,0,0,0.023947
32116,Adm-clerical,Private,United-States,56,Married-civ-spouse,6,Black,Husband,Male,0,0,40,0,0,0.136674
48592,Other-service,Private,United-States,30,Never-married,13,White,Own-child,Female,0,0,24,0,0,0.002522
46390,Transport-moving,Self-emp-not-inc,United-States,46,Married-civ-spouse,7,Black,Husband,Male,0,0,48,0,0,0.106835


In [337]:
false_positive = failure_analysis_table[(failure_analysis_table["actual_label"] == 0) & (failure_analysis_table["predicted_label"] == 1)]

In [341]:
print(false_positive.shape)
false_positive


(552, 15)


,occupation,workclass,native-country,age,marital-status,education-num,race,relationship,sex,capital-gain,capital-loss,hours-per-week,actual_label,predicted_label,class 1 Probabilities
34540,Prof-specialty,Local-gov,United-States,51,Married-civ-spouse,14,White,Husband,Male,0,0,35,0,1,0.692679
7304,Prof-specialty,Private,United-States,23,Married-civ-spouse,13,White,Wife,Female,0,0,40,0,1,0.619446
39470,Exec-managerial,Private,United-States,29,Married-civ-spouse,13,White,Husband,Male,0,1628,47,0,1,0.813264
33567,Sales,Private,United-States,33,Married-civ-spouse,12,White,Husband,Male,0,0,46,0,1,0.500442
44325,Sales,Private,Cuba,43,Married-civ-spouse,10,White,Husband,Male,0,0,50,0,1,0.507258
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35820,NaN,NaN,United-States,67,Married-civ-spouse,13,White,Husband,Male,0,0,10,0,1,0.520186
4236,Prof-specialty,Self-emp-not-inc,United-States,75,Married-civ-spouse,16,White,Husband,Male,4931,0,3,0,1,0.893282
2882,Exec-managerial,Private,United-States,53,Married-civ-spouse,13,Black,Husband,Male,0,0,40,0,1,0.613496
30728,Prof-specialty,Private,United-States,65,Divorced,13,White,Not-in-family,Male,6723,0,40,0,1,0.821967


In [343]:
false_negative = failure_analysis_table[(failure_analysis_table["actual_label"] == 1) & (failure_analysis_table["predicted_label"] == 0)]
false_negative

,occupation,workclass,native-country,age,marital-status,education-num,race,relationship,sex,capital-gain,capital-loss,hours-per-week,actual_label,predicted_label,class 1 Probabilities
9244,Machine-op-inspct,Private,United-States,38,Never-married,12,Black,Not-in-family,Female,10520,0,50,1,0,0.441811
10349,Exec-managerial,Private,United-States,44,Divorced,13,White,Not-in-family,Male,0,0,50,1,0,0.293710
2683,Tech-support,Private,United-States,37,Married-civ-spouse,9,White,Husband,Male,0,0,40,1,0,0.323516
19923,Handlers-cleaners,Private,United-States,28,Married-civ-spouse,7,White,Husband,Male,7688,0,40,1,0,0.426121
28457,Prof-specialty,Local-gov,United-States,60,Widowed,13,White,Unmarried,Female,0,0,60,1,0,0.229496
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36358,Sales,Private,United-States,39,Married-civ-spouse,9,White,Husband,Male,0,0,40,1,0,0.288037
41443,Exec-managerial,Private,United-States,45,Divorced,12,White,Unmarried,Female,0,0,56,1,0,0.120047
25890,Sales,Private,United-States,53,Married-civ-spouse,10,White,Husband,Male,0,0,40,1,0,0.423684
34498,Machine-op-inspct,Private,United-States,57,Married-civ-spouse,9,White,Husband,Male,0,0,40,1,0,0.249639


In [347]:
false_positive.sort_values(by = "class 1 Probabilities", ascending = False).head()

,occupation,workclass,native-country,age,marital-status,education-num,race,relationship,sex,capital-gain,capital-loss,hours-per-week,actual_label,predicted_label,class 1 Probabilities
17039,NaN,NaN,United-States,20,Never-married,10,Black,Other-relative,Male,34095,0,10,0,1,0.997700
3593,Farming-fishing,Self-emp-not-inc,United-States,61,Married-civ-spouse,9,White,Wife,Female,22040,0,40,0,1,0.994992
6232,Prof-specialty,Self-emp-not-inc,United-States,90,Married-civ-spouse,13,White,Husband,Male,10566,0,50,0,1,0.992851
9374,Exec-managerial,Private,United-States,64,Married-civ-spouse,10,White,Wife,Female,10566,0,35,0,1,0.976696
33079,Prof-specialty,Private,United-States,41,Married-civ-spouse,16,White,Husband,Male,0,2051,60,0,1,0.966471


In [348]:
false_negative.sort_values(by = "class 1 Probabilities", ascending = False).head()

,occupation,workclass,native-country,age,marital-status,education-num,race,relationship,sex,capital-gain,capital-loss,hours-per-week,actual_label,predicted_label,class 1 Probabilities
14253,Machine-op-inspct,Private,United-States,36,Married-civ-spouse,11,White,Husband,Male,3103,0,40,1,0,0.499743
28963,Prof-specialty,Private,United-States,55,Married-civ-spouse,10,White,Husband,Male,0,0,40,1,0,0.499729
1448,Exec-managerial,Private,United-States,51,Married-civ-spouse,10,White,Husband,Male,0,0,45,1,0,0.497140
28173,Exec-managerial,State-gov,United-States,49,Never-married,16,White,Not-in-family,Female,0,2258,50,1,0,0.496962
45069,Tech-support,Private,United-States,44,Married-civ-spouse,11,White,Husband,Male,0,0,40,1,0,0.496477


Now, after computing false negative we can see that model incorrectly predict the label for Class 1 with probabilities close to 0.499743. We will check the data to see if reducing the threshold to 0.49, reduce false negatives.

In [350]:
false_negative.sort_values(by = "class 1 Probabilities", ascending = False)

,occupation,workclass,native-country,age,marital-status,education-num,race,relationship,sex,capital-gain,capital-loss,hours-per-week,actual_label,predicted_label,class 1 Probabilities
14253,Machine-op-inspct,Private,United-States,36,Married-civ-spouse,11,White,Husband,Male,3103,0,40,1,0,0.499743
28963,Prof-specialty,Private,United-States,55,Married-civ-spouse,10,White,Husband,Male,0,0,40,1,0,0.499729
1448,Exec-managerial,Private,United-States,51,Married-civ-spouse,10,White,Husband,Male,0,0,45,1,0,0.497140
28173,Exec-managerial,State-gov,United-States,49,Never-married,16,White,Not-in-family,Female,0,2258,50,1,0,0.496962
45069,Tech-support,Private,United-States,44,Married-civ-spouse,11,White,Husband,Male,0,0,40,1,0,0.496477
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15037,Other-service,Self-emp-not-inc,South,46,Divorced,13,Asian-Pac-Islander,Unmarried,Female,0,0,60,1,0,0.008624
1327,Other-service,Private,India,52,Widowed,9,Asian-Pac-Islander,Unmarried,Female,0,0,40,1,0,0.007054
31512,Other-service,Private,United-States,27,Never-married,9,White,Not-in-family,Female,0,0,50,1,0,0.006438
15567,Other-service,State-gov,United-States,61,Widowed,7,White,Unmarried,Female,0,0,32,1,0,0.004450


From what we can observe in the dataframe, reducing  the threshold to 0.40, will reduce many false negatives. Next we evaluate to check if the threshold improves performance and if we should freeze validation choices.

In [364]:
threshhold_045_predictions = (class1_probabilities>=0.45).astype(int)

In [365]:
threshold_45_cm = confusion_matrix(y_val, threshhold_045_predictions)
threshold_45_cm

array([[6767,  665],
       [ 794, 1543]])

In [366]:
threshold_45_accuracy = accuracy_score(y_val, threshhold_045_predictions)
threshold_45_accuracy

0.8506500153546934

In [367]:
threshold_45_precision = precision_score(y_val, threshhold_045_predictions)
threshold_45_precision

0.698822463768116

In [368]:
threshold_45_recall = recall_score(y_val, threshhold_045_predictions)
threshold_45_recall

0.6602481814291827

In [369]:
threshold_45_f1_score = f1_score(y_val, threshhold_045_predictions)
threshold_45_f1_score

0.678987898789879

So moving from 0.50 → 0.45 does this:

  1. catches 101 additional actual >$50K earners: TP 1442 → 1543


  2. misses 101 fewer: FN 895 → 794



  3. creates 113 additional false positives: FP 552 → 665


  4. precision drops about 2.4 percentage points


  5. recall rises about 4.3 percentage points


  6. accuracy barely changes


  7. F1 improves about 1.3 percentage points

  A threshold of 0.45 was selected on the validation set because it improved recall and F1 while causing only a modest reduction in precision and almost no change in overall accuracy.